<a href="https://colab.research.google.com/github/nicosesma/otc_strategies/blob/main/kalshi_Gold_logic_algo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import logging

class KalshiGoldTrader:
    def __init__(self, balance, daily_loss_limit=0.03):
        self.balance = balance
        self.daily_loss_limit = daily_loss_limit # 3% limit
        self.daily_pnl = 0.0
        self.trading_active = True
        self.logger = logging.getLogger("GoldTrader")

    def check_kill_switch(self):
        """Disables trading if daily loss limit is hit."""
        if self.daily_pnl <= -(self.balance * self.daily_loss_limit):
            self.trading_active = False
            self.logger.critical("KILL SWITCH ACTIVATED: Daily loss limit exceeded.")
            return True # Stop trading
        return False

    def get_sentiment_skew(self, model_prob, market_price_cents):
        """
        Calculates if the market is mispricing the event.
        market_price_cents: The current 'Yes' price on Kalshi (0-99).
        """
        market_prob = market_price_cents / 100.0
        skew = model_prob - market_prob
        return skew # Positive skew means market is underpricing 'Yes'

    def run_strategy(self, df, kalshi_price, atr):
        if not self.trading_active or self.check_kill_switch():
            return None

        # 1. Technical Momentum Logic (as defined previously)
        # Assuming df has 'close', 'ema20', 'stoch_rsi'
        model_prob = self.calculate_model_probability(df)

        # 2. Sentiment Skew Analysis
        skew = self.get_sentiment_skew(model_prob, kalshi_price)

        # 3. Decision Engine
        # Only trade if the edge (skew) is significant (> 10%)
        if skew > 0.10:
            size = self.calculate_position_size(atr)
            return {"action": "BUY_YES", "size": size, "reason": f"Skew: {skew:.2f}"}

        elif skew < -0.10:
            return {"action": "BUY_NO", "size": size, "reason": f"Skew: {skew:.2f}"}

        return "HOLD"

    def calculate_position_size(self, atr):
        """Volatility-adjusted sizing based on account risk."""
        risk_per_trade = self.balance * 0.01
        size = risk_per_trade / (atr if atr > 0 else 1)
        return round(size, 2)

    def calculate_model_probability(self, df):
        # Placeholder for your technical indicators logic
        # Should return a float between 0.0 and 1.0
        return 0.65